# VACE-14B Video Editing Demo — Identity Preserving
**GPU**: A100 40GB (Colab Pro)
**Time**: ~20 min

CelebV-HQ face → VACE-14B editing with face masking → composited output

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
!pip install -q "diffusers>=0.33.0" transformers accelerate peft
!pip install -q imageio imageio-ffmpeg safetensors sentencepiece protobuf ftfy
!pip install -q opencv-python-headless

## 1. Download a sample CelebV-HQ video

In [ ]:
import os, subprocess
os.makedirs('input', exist_ok=True)
os.makedirs('results', exist_ok=True)

# Download a sample talking head video from CelebV-HQ (public sample)
# Using a royalty-free talking head video as substitute
!wget -q -O input/sample_face.mp4 "https://huggingface.co/datasets/Wild-Heart/Disney-VideoGeneration-Dataset/resolve/main/videos/0001.mp4" 2>/dev/null || true

# If that doesn't work, generate a reference frame
if not os.path.exists('input/sample_face.mp4') or os.path.getsize('input/sample_face.mp4') < 1000:
    print('Using synthetic reference instead')
    USE_SYNTHETIC = True
else:
    USE_SYNTHETIC = False
    # Extract frames
    !ffmpeg -y -i input/sample_face.mp4 -ss 1 -t 2 -vf "fps=8,scale=832:480" input/frame_%03d.png -loglevel quiet
    !ffmpeg -y -i input/sample_face.mp4 -ss 2 -frames:v 1 -vf "scale=832:480" input/reference.png -loglevel quiet
    frames = sorted([f'input/{f}' for f in os.listdir('input') if f.startswith('frame_')])
    print(f'Extracted {len(frames)} frames + reference')

## 2. Load VACE-14B Pipeline

In [ ]:
import torch
from diffusers import WanVACEPipeline, AutoencoderKLWan
from diffusers.schedulers.scheduling_unipc_multistep import UniPCMultistepScheduler
from diffusers.utils import export_to_video
from PIL import Image, ImageDraw, ImageFilter
import numpy as np
import time

model_id = "Wan-AI/Wan2.1-VACE-14B-diffusers"

# VAE MUST be fp32 to avoid NaN
vae = AutoencoderKLWan.from_pretrained(model_id, subfolder="vae", torch_dtype=torch.float32)
pipe = WanVACEPipeline.from_pretrained(model_id, vae=vae, torch_dtype=torch.bfloat16)
pipe.scheduler = UniPCMultistepScheduler.from_config(pipe.scheduler.config, flow_shift=3.0)
pipe.enable_model_cpu_offload()

print(f'VACE-14B loaded')
print(f'GPU: {torch.cuda.get_device_name(0)}')

## 3. Create Face Masks (OpenCV detection)

In [ ]:
import cv2

def detect_and_mask_faces(frames, expand=0.35, blur=15):
    """Detect faces in frames, return masks where black=face(preserve), white=edit."""
    cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')
    masks = []
    bboxes = []
    for frame in frames:
        w, h = frame.size
        gray = cv2.cvtColor(np.array(frame), cv2.COLOR_RGB2GRAY)
        faces = cascade.detectMultiScale(gray, 1.2, 5, minSize=(50, 50))
        mask = Image.new('L', (w, h), 255)  # white = edit
        draw = ImageDraw.Draw(mask)
        if len(faces) > 0:
            for (fx, fy, fw, fh) in faces:
                x1 = max(0, int(fx - fw * expand))
                y1 = max(0, int(fy - fh * expand))
                x2 = min(w, int(fx + fw + fw * expand))
                y2 = min(h, int(fy + fh + fh * expand))
                draw.rectangle([x1, y1, x2, y2], fill=0)  # black = preserve
                bboxes.append((x1, y1, x2, y2))
        else:
            # fallback: center region
            cx, cy = w//2, h//3
            draw.rectangle([cx-120, cy-150, cx+120, cy+150], fill=0)
            bboxes.append((cx-120, cy-150, cx+120, cy+150))
        mask = mask.filter(ImageFilter.GaussianBlur(radius=blur))
        masks.append(mask)
    return masks, bboxes

def composite_face(orig_frames, gen_frames, masks):
    """Paste original face back onto generated output."""
    result = []
    for orig, gen, mask in zip(orig_frames, gen_frames, masks):
        if hasattr(gen, 'size'):
            size = gen.size
        else:
            size = (832, 480)
        orig_r = orig.resize(size)
        mask_r = mask.resize(size)
        o = np.array(orig_r).astype(float)
        g = np.array(gen).astype(float) if hasattr(gen, '__array__') else np.array(gen).astype(float)
        m = np.array(mask_r).astype(float) / 255.0  # 0=face, 1=edit
        comp = o * (1 - m[:,:,None]) + g * m[:,:,None]
        result.append(Image.fromarray(comp.astype(np.uint8)))
    return result

print('Face detection + compositing functions ready')

## 4. Generate a Reference Face (if no CelebV-HQ)

In [ ]:
if USE_SYNTHETIC:
    # Generate a reference person first using T2V
    from diffusers import WanPipeline
    pipe_gen = WanPipeline.from_pretrained('Wan-AI/Wan2.1-T2V-1.3B-Diffusers', torch_dtype=torch.float16)
    pipe_gen.to('cuda')
    ref_out = pipe_gen(
        prompt='A young woman with brown hair looking at camera, neutral expression, studio lighting, portrait, photorealistic, 4K',
        num_frames=17, height=480, width=832,
        num_inference_steps=30, guidance_scale=5.0,
        generator=torch.Generator('cuda').manual_seed(42),
    )
    src_frames = ref_out.frames[0]
    src_frames[0].save('input/reference.png')
    export_to_video(src_frames, 'input/source_video.mp4', fps=8)
    del pipe_gen; torch.cuda.empty_cache()
    print(f'Generated {len(src_frames)} reference frames')
else:
    src_frames = [Image.open(f).convert('RGB').resize((832, 480)) for f in frames]
    print(f'Loaded {len(src_frames)} CelebV-HQ frames')

# Trim to 4k+1
num_f = ((len(src_frames) - 1) // 4) * 4 + 1
src_frames = src_frames[:num_f]
print(f'Using {num_f} frames')

# Detect faces
face_masks, face_bboxes = detect_and_mask_faces(src_frames)
print(f'Detected faces in {len(face_bboxes)} regions')

# Show input + mask
from IPython.display import display
print('Input reference:')
display(src_frames[0].resize((416, 240)))
print('Face mask (black=preserve, white=edit):')
display(face_masks[0].resize((416, 240)))

## 5. Run Edits — Face Preserved

In [ ]:
NEGATIVE = ('Bright tones, overexposed, static, blurred details, subtitles, style, works, '
           'paintings, images, static, overall gray, worst quality, low quality, JPEG compression '
           'residue, ugly, incomplete, extra fingers, poorly drawn hands, poorly drawn faces, '
           'deformed, disfigured, misshapen limbs, fused fingers, still picture, messy background, '
           'three legs, many people in the background, walking backwards')

ref_image = src_frames[0]  # Use first frame as reference

edits = [
    {'name': 'hair_to_blonde', 'prompt': 'A person with bright blonde hair, natural look, photorealistic, same expression, 4K video'},
    {'name': 'background_beach', 'prompt': 'A person at a tropical beach with palm trees and turquoise ocean, golden sunset light, photorealistic, cinematic, 4K'},
    {'name': 'formal_outfit', 'prompt': 'A person wearing an elegant dark formal suit with white shirt, professional photoshoot, studio lighting, photorealistic, 4K'},
]

for edit in edits:
    print(f"\n{'='*50}")
    print(f"Edit: {edit['name']}")
    print(f"{'='*50}")
    torch.cuda.reset_peak_memory_stats()
    t0 = time.time()

    # Prepare conditioning: original pixels where face is, gray where we edit
    cond_frames = []
    for frame, mask in zip(src_frames, face_masks):
        f_np = np.array(frame).astype(float)
        m_np = np.array(mask).astype(float) / 255.0
        gray = np.full_like(f_np, 128.0)
        blended = f_np * (1 - m_np[:,:,None]) + gray * m_np[:,:,None]
        cond_frames.append(Image.fromarray(blended.astype(np.uint8)))

    try:
        out = pipe(
            prompt=edit['prompt'],
            negative_prompt=NEGATIVE,
            video=cond_frames,
            mask=face_masks,
            reference_images=[ref_image],
            height=480, width=832,
            num_frames=num_f,
            num_inference_steps=30,
            guidance_scale=5.0,
            generator=torch.Generator('cpu').manual_seed(42),
        )
        dt = time.time() - t0
        peak = torch.cuda.max_memory_allocated() / 1e9

        # Raw output
        raw_path = f'results/{edit["name"]}_raw.mp4'
        export_to_video(out.frames[0], raw_path, fps=8)

        # Composited (paste original face back)
        comp_frames = composite_face(src_frames, out.frames[0], face_masks)
        comp_path = f'results/{edit["name"]}_composited.mp4'
        export_to_video(comp_frames, comp_path, fps=8)

        # Save comparison frames
        comp_frames[0].save(f'results/{edit["name"]}_first.png')
        comp_frames[-1].save(f'results/{edit["name"]}_last.png')

        print(f'Done: {dt:.1f}s, VRAM: {peak:.1f} GB')
        print(f'Files: {raw_path}, {comp_path}')

    except Exception as e:
        import traceback
        print(f'FAILED: {e}')
        traceback.print_exc()
        torch.cuda.empty_cache()

## 6. Side-by-Side Comparison

In [ ]:
import base64
from IPython.display import HTML, display

def show_video(path, label):
    if not os.path.exists(path) or os.path.getsize(path) < 1000:
        return f'<div><b>{label}</b><br>Failed</div>'
    with open(path, 'rb') as f:
        b64 = base64.b64encode(f.read()).decode()
    return f'''<div style="display:inline-block;margin:8px;text-align:center">
        <b>{label}</b><br>
        <video width="380" controls autoplay loop muted>
            <source src="data:video/mp4;base64,{b64}" type="video/mp4">
        </video></div>'''

# Show input
display(HTML('<h2>Input Reference</h2>'))
display(src_frames[0].resize((416, 240)))

# Show each edit: raw vs composited
for edit in edits:
    display(HTML(f'<h3>Edit: {edit["name"]}</h3>'))
    html = show_video(f'results/{edit["name"]}_raw.mp4', 'Raw VACE Output')
    html += show_video(f'results/{edit["name"]}_composited.mp4', 'Composited (Face Preserved)')
    display(HTML(html))

In [ ]:
# Download all results
!zip -r vace_demo_results.zip results/ input/reference.png
from google.colab import files
files.download('vace_demo_results.zip')